# Example: E. coli Drug Time-Course Phenotyping

Analyze the temporal dynamics of morphological changes in *E. coli* treated with various antibiotics. This workflow demonstrates how to capture time-dependent drug effects through deep learning and classical profiling.

**Dataset:** [E. coli CLSM](https://bbbc.broadinstitute.org/) (Confocal Microscopy)

**Key Steps:**
1. **Metadata Extraction:** Parse conditions (drug, time, replicate) directly from directory structures.
2. **Feature Extraction:** Compute DINOv2 deep embeddings and classical morphological properties.
3. **Temporal Dynamics:** Analyze how antibiotic treatments shift bacterial phenotypes over time through clustering and distance analysis.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Define the paths to the *E. coli* image dataset and masks. This example utilizes a directory structure where metadata is implicitly encoded in the file paths.

In [ ]:
DATA_DIR = "/datasets/Ecoli_CLSM/preprocessed_BB/"
MASK_DIR = "/datasets/Ecoli_CLSM/preprocessed_masks_BB/"

## 2. File Discovery & Metadata Extraction

We use `get_metadata_from_path` to automatically parse 'drug', 'replicate', and 'time' from the directory structure, enabling efficient metadata management without external CSV files.

In [ ]:
from phenome.utils import get_metadata_from_path

# Define the path pattern to extract metadata
path_pattern = ".../(drug)_#(replicate)/(time)/(img_name)/(crop_name).*"

file_df = pheno.find_files(
    DATA_DIR,
    mask_dir=MASK_DIR,
    extensions=[".tif", ".tiff"],
    metadata_fn=get_metadata_from_path(path_pattern),
)

# Clean up metadata values
file_df["time"] = file_df["time"].replace({"0_min": "00_min", "2_min": "02_min", "5_min": "05_min"})
file_df["drug"] = file_df["drug"].str.replace(r"(?i)_replicate$", "", regex=True)
pheno.set_file_df(file_df)

print(f"Drugs found: {sorted(file_df['drug'].unique())}")
print("Available time options:", sorted(file_df["time"].unique()))
file_df.head()

## 3. Feature Extraction

We extract DINOv2 embeddings and compute extended morphological properties for selected drugs and time points to analyze the antibiotic-induced phenotypic shifts.

In [ ]:
eval_drugs = ["Control", "Chloramphenicol", "Azide", "MP265", "Mecillinam", "Nalidixate", "Rifampicin"]
eval_times = ["60_min"]

# Extract embeddings with filters
pheno.process_images(
    wrapper,
    filters={"drug": eval_drugs, "time": eval_times},
    batch_size=328,
    channel_mode="split",
    resize_size=224,  # Target size for resizing images
    pad_size=168,  # Pad small images to 168x168 before resizing to 224x224
)

In [ ]:
# Compute properties
df_properties = pheno.compute_properties(
    property_preset="complete",
    n_jobs=8,
)

## 4. Visualization

Visualize the centroids of different drug treatments in the phenospace to observe how various antibiotics uniquely alter bacterial morphology over time.

In [ ]:
pheno.plot_centroids(
    group_by=["drug"],
    method="tsne",
    show_points=True,
)

## 5. Interactive Exploration

Launch the interactive explorer to visually inspect individual bacteria and explore the relationship between drug treatment and morphological response.

In [ ]:
e = pheno.create_interactive_explorer()